# 03 · 내 문서 Grounding

**목표**: AI Core vector grounding 저장소의 문서를 검색하고, 검색 청크를 확인한 뒤 근거 기반 답변을 생성합니다.

공개 Help 저장소와 달리 data_repository_type을 vector로 바꾸고 collection ID를 지정합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: 내 문서 Grounding 질의</h2>
<p style="margin: 7px 0;">선택된 vector collection에 질문을 보냅니다. UI를 실행하기 전에 위 셀에서 collection ID가 정상적으로 선택됐는지 확인하세요.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import os
import sys
import gradio as gr

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
from lab_common.config import load_settings
from lab_common.ai_core import apply_ai_core_environment, orchestration_url
from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig
from gen_ai_hub.orchestration_v2.models.document_grounding import (
    DocumentGroundingFilter, DocumentGroundingConfig, DocumentGroundingPlaceholders, GroundingModuleConfig)
from gen_ai_hub.orchestration_v2.models.template import Template, PromptTemplatingModuleConfig
from gen_ai_hub.orchestration_v2.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.service import OrchestrationService

def run_my_docs_ui(user_query: str) -> str:
    try:
        settings = load_settings()
        apply_ai_core_environment(settings)
        repo_id = os.getenv('GROUNDING_REPOSITORY_ID')
        if not repo_id:
            return 'GROUNDING_REPOSITORY_ID가 없습니다. 루트 .env에 사용할 vector collection ID를 설정하세요.'
        template = Template(template=[
            SystemMessage(content='Answer only from the provided context. Say when context is insufficient.'),
            UserMessage(content='Request: {{?user_query}}\nContext: {{?grounding_response}}'),
        ])
        grounding = GroundingModuleConfig(config=DocumentGroundingConfig(
            filters=[DocumentGroundingFilter(id='MyDocs', data_repository_type='vector', data_repositories=[repo_id])],
            placeholders=DocumentGroundingPlaceholders(input=['user_query'], output='grounding_response'),
        ))
        prompt = PromptTemplatingModuleConfig(
            prompt=template, model=LLMModelDetails(name=settings.llm_model, params={'max_tokens': 1000})
        )
        service = OrchestrationService(
            api_url=orchestration_url(settings),
            config=OrchestrationConfig(modules=ModuleConfig(prompt_templating=prompt, grounding=grounding)),
        )
        result = service.run(placeholder_values={'user_query': user_query})
        return result.final_result.choices[0].message.content
    except Exception as error:
        return f'내 문서 Grounding 호출 실패: {type(error).__name__}: {error}'

my_docs_ui = gr.Interface(
    fn=run_my_docs_ui,
    inputs=gr.Textbox(label='Question', lines=3),
    outputs=gr.Textbox(label='Grounded answer', lines=10),
    title='My Documents Grounding Playground',
    description='루트 .env의 GROUNDING_REPOSITORY_ID를 사용합니다.',
)
my_docs_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", height=900)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>UI에서 한 번 실행해 결과를 확인한다.</li>
<li>결과가 기대와 다르면 환경 설정과 모델/저장소 이름을 먼저 점검한다.</li>
<li>아래 코드는 같은 기능의 구성 요소를 자세히 살펴보는 선택 심화 자료다.</li>
</ul>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI가 사용하는 구성과 동작을 단계별로 확인하는 자료입니다. 기본 실습에서는 실행하지 않아도 됩니다.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. AI Core REST와 SDK 준비</h2>
<p style="margin: 7px 0;">이 노트북은 REST로 collection과 검색 결과를 확인하고 SDK Orchestration으로 최종 답변을 생성합니다. 모두 같은 .env의 AI Core 인증을 사용합니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
from lab_common.config import load_settings
from lab_common.ai_core import api_headers, apply_ai_core_environment, orchestration_url

settings = load_settings()
apply_ai_core_environment(settings)
ORCH = orchestration_url(settings)

import os
import requests

from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig
from gen_ai_hub.orchestration_v2.models.document_grounding import (
    DocumentGroundingFilter, DocumentGroundingConfig,
    DocumentGroundingPlaceholders, GroundingModuleConfig)
from gen_ai_hub.orchestration_v2.models.template import Template, PromptTemplatingModuleConfig
from gen_ai_hub.orchestration_v2.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.service import OrchestrationService

print("=" * 56)
print("Step 5 · 내 문서 그라운딩")
print("=" * 56)

base = os.getenv("AICORE_BASE_URL").rstrip("/")
h = api_headers(settings)
DG = f"{base}/v2/lm/document-grounding"

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 저장소 선택</h2>
<p style="margin: 7px 0;">GROUNDING_REPOSITORY_ID가 있으면 그 저장소를 사용합니다. 없으면 문서가 들어 있는 첫 collection을 읽기 전용으로 찾습니다. 교육 환경에서는 강사가 공유한 ID를 .env에 지정하는 편이 가장 재현성이 좋습니다.</p>
  </div>
</div>


In [ ]:
# 1) 저장소 id 결정 (env 우선, 없으면 문서 있는 첫 저장소)
repo_id = os.getenv("GROUNDING_REPOSITORY_ID")
if not repo_id:
    cols = requests.get(f"{DG}/vector/collections", headers=h, timeout=30).json()
    cols = cols.get("resources", cols) if isinstance(cols, dict) else cols
    for c in cols:
        cid = c.get("id")
        n = requests.get(f"{DG}/vector/collections/{cid}/documents",
                         headers=h, timeout=30).json().get("count", 0)
        if n and int(n) > 0:
            repo_id = cid
            print(f"  (자동 선택) title={c.get('title')} docs={n}")
            break
assert repo_id, "문서가 든 저장소를 찾지 못함"
print("  저장소 id:", repo_id)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 질문 설정</h2>
<p style="margin: 7px 0;">Q 환경변수로 질문을 바꿀 수 있습니다. 질문은 저장소 문서의 실제 내용과 관련 있게 작성해야 검색 결과를 관찰하기 쉽습니다.</p>
  </div>
</div>


In [ ]:
# 2) 질문 — 저장소 내용에 맞춰 (procurement policy 가정 + 일반 질문 둘 다 시도)
QUESTION = os.getenv("Q", "What does this document say? Summarize the key points.")

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. 검색된 청크 먼저 보기</h2>
<p style="margin: 7px 0;">LLM 답변 전에 retrieval/search 결과와 score를 확인합니다. 이는 모델 답변의 근거를 살피는 단계입니다. 점수 절대값보다 질문을 바꿨을 때 어떤 청크가 선택되는지 비교하세요.</p>
  </div>
</div>


In [ ]:
# 2a) 먼저 검색된 청크 보기 (retrieval/search)
print(f"\n[1] 검색된 청크 보기 — '{QUESTION}'")
body = {"query": QUESTION,
        "filters": [{"id": "f", "dataRepositoryType": "vector", "dataRepositories": [repo_id]}]}
r = requests.post(f"{DG}/retrieval/search", headers=h, json=body, timeout=60).json()
found = 0
for res in r.get("results", []):
    for rr in res.get("results", []):
        for doc in rr.get("dataRepository", {}).get("documents", []):
            for ch in doc.get("chunks", []):
                found += 1
                score = ch.get("searchScores", {}).get("aggregatedScore", {}).get("value", 0)
                print(f"  [{found}] score={score:.3f}  {ch.get('content','')[:120]}")
if not found:
    print("  (검색된 청크 없음)")

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. Vector Grounding으로 답변 생성</h2>
<p style="margin: 7px 0;">마지막 단계는 공개 Grounding과 거의 같습니다. 필터만 vector와 선택한 collection ID로 바꾸어 내 문서 근거를 prompt에 주입합니다.</p>
  </div>
</div>


In [ ]:
# 3) 그라운딩 질의 — 모듈 4와 '필터만' 다름
print(f"\n[2] 그라운딩 LLM 답변 — '{QUESTION}'")
llm = LLMModelDetails(name=settings.llm_model, params={"max_tokens": 1000})
template = Template(template=[
    SystemMessage(content="You are a helpful assistant. Answer only from the provided context."),
    UserMessage(content="""Request: {{{?user_query}}}
        Context: {{{?grounding_response}}}
        """),
])
filters = [DocumentGroundingFilter(
    id="MyDocs", data_repository_type="vector", data_repositories=[repo_id])]
grounding_config = GroundingModuleConfig(config=DocumentGroundingConfig(
    filters=filters,
    placeholders=DocumentGroundingPlaceholders(input=["user_query"], output="grounding_response")))
prompt_template = PromptTemplatingModuleConfig(prompt=template, model=llm)
config = OrchestrationConfig(modules=ModuleConfig(
    prompt_templating=prompt_template, grounding=grounding_config))
service = OrchestrationService(api_url=ORCH, config=config)
resp = service.run(placeholder_values={"user_query": QUESTION})
print("  응답:", resp.final_result.choices[0].message.content[:500])

print("\n✅ Step 5 완료 — 모듈 4 대비 '필터만' 바꿔 내 문서 기반 답변.")